# Phase 24 — Relational Interface and Co-adaptation (Phase 16)
## NeuroForge Experimental Research

Central observation (Phase 15): the dedicated Graph specialist exploits relational structure strongly (≈84–93% R, ≈32–47pp destruction drops) while the JointCo embedded path does not (≈5pp drops). Question: WHY? Localize among input/interface, co-adaptation, post-branch loss, training signal, or effective computation. Localization only — no architecture is earned until the discrepancy is understood.

## 1. Environment verification

In [1]:
import platform, torch, neuroforge
print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'NeuroForge: {neuroforge.__file__}')

Python: 3.13.14
PyTorch: 2.13.0+cpu
NeuroForge: C:\Projects\NN\src\neuroforge\__init__.py


## 2. Exact baselines (16A — executed, never historical numbers)

In [2]:
import json
p16 = json.load(open('../results/metrics/phase16_relational_interface/summary.json', encoding='utf-8'))
for k in ('graph_perf_mean','baseline_perf_mean','depth3_perf_mean','rel_focused_perf_mean','rel_first_perf_mean'):
    d = p16[k]
    print(f"{k:>22}: " + '  '.join(f"{f}={d.get(f,0)*100:.1f}%" for f in ('F','R','C','FR','RC','FC','FRC')))
print('reproduction vs Phase 15:', p16['baseline_check_vs_phase15'])

       graph_perf_mean: F=50.0%  R=89.2%  C=54.2%  FR=49.2%  RC=52.2%  FC=48.9%  FRC=52.2%
    baseline_perf_mean: F=100.0%  R=57.8%  C=98.9%  FR=76.9%  RC=53.9%  FC=49.4%  FRC=79.4%
      depth3_perf_mean: F=100.0%  R=62.5%  C=99.2%  FR=76.7%  RC=53.6%  FC=50.0%  FRC=80.0%
 rel_focused_perf_mean: F=100.0%  R=60.8%  C=52.2%  FR=72.8%  RC=46.7%  FC=46.4%  FRC=50.3%
   rel_first_perf_mean: F=100.0%  R=62.8%  C=98.9%  FR=74.7%  RC=54.2%  FC=48.9%  FRC=78.9%
reproduction vs Phase 15: {'phase15_available': True, 'reproduced': True, 'deltas': {'baseline_R': 0.0, 'baseline_RC': 0.0, 'baseline_FRC': 0.0, 'depth3_R': 0.0, 'depth3_RC': 0.0, 'depth3_FRC': 0.0}}


## 3. Relational input equivalence (16B — same raw batch, different encoders)

In [3]:
import csv
for row in csv.DictReader(open('../results/metrics/phase16_relational_interface/input_equivalence.csv')):
    if row['seed'] == str(p16['per_seed_results'][0]['seed']):
        print(row)
print('Question answered: are we comparing the same relational problem at the input?')

{'seed': '11', 'representation': 'joint', 'global_mean': '-0.025584666058421135', 'global_std': '0.3497820794582367', 'mean_abs': '0.28099262714385986', 'marker_max': '1.0'}
{'seed': '11', 'representation': 'native', 'global_mean': '0.003998559433966875', 'global_std': '0.38860446214675903', 'mean_abs': '0.3076719343662262', 'marker_max': '1.0'}
{'seed': '11', 'representation': 'comparison', 'global_mean': '0.02958322549238801', 'global_std': '0.9000979492771217', 'mean_abs': '1.3160374164581299', 'marker_max': '1.0'}
Question answered: are we comparing the same relational problem at the input?


## 4. Graph-on-Joint-input (16C — THE critical experiment: same input, different computation)

In [4]:
c = p16['aggregates']['computation']
print(f"embedded branch + head R: {c['embedded_R']*100:.1f}%")
print(f"graph-on-joint-input + head R: {c['graph_on_joint_R']*100:.1f}%")
print(f"same-input gap: {c['graph_on_joint_minus_embedded_R']*100:+.1f}pp")
print(f"fresh graph diagnostic R: {p16['aggregates']['graph_diagnostic']['diagnostic_R']*100:.1f}%")
print('If graph-on-joint >> embedded: computation suspect. If similar: input suspect.')

embedded branch + head R: 69.7%
graph-on-joint-input + head R: 60.6%
same-input gap: -9.2pp
fresh graph diagnostic R: 57.8%
If graph-on-joint >> embedded: computation suspect. If similar: input suspect.


## 5. Joint-input Graph probe (16D — can ANY Graph computation exploit the joint input?)

In [5]:
g = p16['aggregates']['graph_diagnostic']
print(f"fresh graph diagnostic R: {g['diagnostic_R']*100:.1f}% (probe {g['diagnostic_probe_R']*100:.1f}%)")
print(f"native Graph R (reference): {p16['graph_perf_mean']['R']*100:.1f}%")
print(f"native-minus-diagnostic: {g['native_minus_diagnostic_R']*100:+.1f}pp")

fresh graph diagnostic R: 57.8% (probe 78.3%)
native Graph R (reference): 89.2%
native-minus-diagnostic: +31.4pp


## 6. Pre/post relational analysis (16E — probe + usability + causal dependence)

In [6]:
st = p16['aggregates']['stages']
print(f"pre-rel probe {st['pre_probe_R']*100:.1f}% / head {st['pre_R']*100:.1f}% / drop {st['pre_drop_R']*100:+.1f}pp")
print(f"post-rel probe {st['post_probe_R']*100:.1f}% / head {st['post_R']*100:.1f}% / drop {st['post_drop_R']*100:+.1f}pp")
print(f"post-fusion probe {st['post_fusion_probe_R']*100:.1f}% / head {st['post_fusion_R']*100:.1f}%")
print(f"native-input probe {st['native_input_probe_R']*100:.1f}% / final R {st['final_R']*100:.1f}%")

pre-rel probe 57.2% / head 52.2% / drop +0.0pp
post-rel probe 85.0% / head 68.1% / drop +15.3pp
post-fusion probe 83.9% / head 60.8%
native-input probe 59.4% / final R 62.5%


## 7. Branch-freeze co-adaptation (16F — is joint optimization suppressive?)

In [7]:
co = p16['aggregates']['coadaptation']
print(f"joint (depth3) R: {co['joint_R']*100:.1f}%")
print(f"rel-focused R: {co['relfocused_R']*100:.1f}% ({co['relfocused_minus_joint_R']*100:+.1f}pp)")
print(f"rel-first R: {co['relfirst_R']*100:.1f}% ({co['relfirst_minus_joint_R']*100:+.1f}pp)")

joint (depth3) R: 62.5%
rel-focused R: 60.8% (-1.7pp)
rel-first R: 62.8% (+0.3pp)


## 8. Gradient/signal diagnosis (16G — diagnostic instrumentation only)

In [8]:
gr = p16['aggregates']['gradients']
print('status:', gr.get('status'))
print('group means:', {k: round(v,4) for k,v in gr.get('group_means', {}).items()})
print('observed:', gr.get('observed'))

status: VERIFIED
group means: {'relational': 0.18, 'feature': 0.3459, 'contextual': 0.5211, 'fusion': 0.1535, 'encoder': 0.7435, 'head': 1.0368}
observed: rel share 6.0% of grad-norm mass


## 9. Branch ablation on the depth-3 candidate (16H)

In [9]:
import csv
best = p16['best_cond']
for row in csv.DictReader(open('../results/metrics/phase16_relational_interface/branch_ablation.csv')):
    if row['seed'] == str(p16['per_seed_results'][0]['seed']) and row['condition'] == ('depth3' if best != 'depth3' else 'depth3'):
        print(f"{row['combination']:>6}: R={float(row['R'])*100:.1f}% RC={float(row['RC'])*100:.1f}% FRC={float(row['FRC'])*100:.1f}%")
print('Question: does relational computation degrade specifically when other branches are present?')

     F: R=55.8% RC=51.7% FRC=72.5%
     R: R=53.3% RC=49.2% FRC=72.5%
     C: R=47.5% RC=51.7% FRC=80.0%
   F+R: R=61.7% RC=50.0% FRC=70.8%
   R+C: R=54.2% RC=51.7% FRC=80.0%
   F+C: R=55.8% RC=51.7% FRC=80.0%
 F+R+C: R=60.0% RC=51.7% FRC=80.0%
Question: does relational computation degrade specifically when other branches are present?


## 10. Representation compatibility (16I — raw vs joint-pre vs native input)

In [10]:
import csv
for row in csv.DictReader(open('../results/metrics/phase16_relational_interface/graph_on_joint_input.csv')):
    if row['seed'] == str(p16['per_seed_results'][0]['seed']):
        print(f"{row['condition']:>26}: R={float(row['R'])*100:.1f}% RC={float(row['RC'])*100:.1f}% FRC={float(row['FRC'])*100:.1f}%")

           embedded_branch: R=80.0% RC=46.7% FRC=55.0%


      graph_on_joint_input: R=77.5% RC=45.8% FRC=56.7%
    graph_diagnostic_fresh: R=67.5% RC=44.2% FRC=50.8%
         graph_native_full: R=90.8% RC=51.7% FRC=48.3%
 compat_rep3_head_protocol: R=62.5% RC=40.8% FRC=47.5%


## 11. Destruction as causal metric (16J) + composition (16K)

In [11]:
se = p16['aggregates']['sensitivity']
print(f"baseline drop {se['baseline_drop_R']*100:+.1f}pp vs {se['candidate']} drop {se['candidate_drop_R']*100:+.1f}pp")
cp = p16['aggregates']['compositional']
print(f"strongest ({cp['condition']}): R {cp['R_gain']*100:+.1f}pp, RC {cp['RC_gain']*100:+.1f}pp, FRC {cp['FRC_gain']*100:+.1f}pp")
print('High R + high drop differs materially from similar R + low drop.')

baseline drop +5.3pp vs rel_first drop +11.1pp
strongest (rel_first): R +5.0pp, RC +0.3pp, FRC -0.6pp
High R + high drop differs materially from similar R + low drop.


## 12. Compute, latency, seeds

In [12]:
import csv
for _name in ('compute.csv','latency.csv','seed_results.csv'):
    print(f'--- {_name} ---')
    for i, row in enumerate(csv.DictReader(open('../results/metrics/phase16_relational_interface/' + _name))):
        if i < 6:
            print(' ', dict(row))
print('stability:', {k: {c: f'{v*100:.1f}pp' for c,v in d.items()} for k,d in p16['aggregates']['seed_stability'].items()})

--- compute.csv ---
  {'seed': '11', 'condition': 'graph', 'params': '3866', 'rel_params': '1776'}
  {'seed': '11', 'condition': 'baseline', 'params': '6464', 'rel_params': '1776'}
  {'seed': '11', 'condition': 'depth3', 'params': '10016', 'rel_params': '5328'}
  {'seed': '11', 'condition': 'rel_focused', 'params': '10016', 'rel_params': '5328'}
  {'seed': '11', 'condition': 'rel_first', 'params': '10016', 'rel_params': '5328'}
  {'seed': '23', 'condition': 'graph', 'params': '3866', 'rel_params': '1776'}
--- latency.csv ---
  {'seed': '11', 'condition': 'graph', 'latency_us': '35.96166665374767', 'throughput_per_s': '27807.38750593438'}
  {'seed': '11', 'condition': 'baseline', 'latency_us': '68.29949999882956', 'throughput_per_s': '14641.3956180812'}
  {'seed': '11', 'condition': 'depth3', 'latency_us': '125.36266666150671', 'throughput_per_s': '7976.856480726534'}
  {'seed': '11', 'condition': 'rel_focused', 'latency_us': '212.27308333133502', 'throughput_per_s': '4710.912868962805'

## 13. H1–H8 (re-derived programmatically)

In [13]:
from neuroforge.evaluation.phase16_metrics import build_phase16_hypotheses
hyps = build_phase16_hypotheses(p16['aggregates'])
for h in sorted(hyps):
    print(f"{h}: {hyps[h]['status']}")
    print(f"    {hyps[h]['evidence']}")
assert all(hyps[h]['status'] == p16['hypotheses'][h]['status'] for h in hyps)
print('stored verdicts match fresh derivation: OK')

H1: INCONCLUSIVE
    Same-input gap -9.2pp, native lead +28.6pp: ambiguous.
H2: NOT SUPPORTED
    Same-input computation gap -9.2pp on R; computation is not the differentiator.
H3: NOT SUPPORTED
    Relational-focused training vs joint training on R: -1.7pp. Rel-first: +0.3pp.
H4: INCONCLUSIVE
    Pre-relational probe 57.2%, native 59.4%.
H5: PARTIALLY SUPPORTED
    Post-relational probe 85.0% but final R 62.5%: conversion loss downstream.
H6: NOT SUPPORTED
    RC +0.3pp, FRC -0.6pp: isolated capability did not transfer.
H7: SUPPORTED
    Causal sensitivity increases by +5.8pp on R.
H8: NOT SUPPORTED
    Ceiling delta +0.1pp mixed mean.
stored verdicts match fresh derivation: OK


## 14. Final CASE + evidence-backed recommendation (programmatic)

In [14]:
from neuroforge.evaluation.phase16_metrics import select_phase16_case, recommendation_for_case
case, label = select_phase16_case(hyps, p16['aggregates']['stages'])
print(f'Programmatic verdict: {case} — {label}')
assert case == p16['verdict_case']
print(f'Minimal intervention: {p16["minimal_intervention"]["intervention"]} ({p16["minimal_intervention"]["outcome"]})')
print(f'Recommendation: {recommendation_for_case(case)}')
print()
print('Evidence (loaded, not typed):')
print(f"  same-input gap {p16['aggregates']['computation']['graph_on_joint_minus_embedded_R']*100:+.1f}pp; "+
      f"rel-focused {p16['aggregates']['coadaptation']['relfocused_minus_joint_R']*100:+.1f}pp; "+
      f"pre/post probes {p16['aggregates']['stages']['pre_probe_R']*100:.1f}/{p16['aggregates']['stages']['post_probe_R']*100:.1f}%; "+
      f"grad status {p16['aggregates']['gradients']['status']}")

Programmatic verdict: CASE E — Relational information exists but remains difficult to convert into task-usable mixed reasoning


Minimal intervention: none (No architectural intervention (localization only))
Recommendation: Treat mixed composition (RC/FRC) as the open problem; isolated R gains do not transfer.

Evidence (loaded, not typed):
  same-input gap -9.2pp; rel-focused -1.7pp; pre/post probes 57.2/85.0%; grad status VERIFIED
